# Resume P1 after the worker MemoryError

Upload this **single notebook** to `~/P1` in the existing native Jupyter environment. It resumes the original `feedback-native-20261024-v1` results. **Keep the existing results/checkpoints and original package.** No experiment source, seed, sample size or comparison is changed.

The recovery adapter uses one fresh spawned process per original chunk, up to eight concurrent workers. It is designed for the server's Python 3.10. Long-lived native workers are replaced by a bounded-lifetime execution method; the exact memory retention cause has not been proved.

Before resuming, it repeats the selected original smokes in a separate validation folder and requires exact agreement with saved smoke compact results. Existing successful full-study checkpoints are hash-checked and reused. Local engineering tests pass; **native recovery validation must happen on your server**.

Each independently allocated eight-core session may run a separate study with eight workers. To split these two studies, select `FAMILY="timing"` in one session and `FAMILY="application"` in the other. Do not run `"all"` alongside either or start two copies of the same family against the same output folder.


In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.home() / "P1"
RUN_ROOT = PROJECT_DIR / "feedback-native-20261024-v1"  # Keep the existing run
PACKAGE = RUN_ROOT / "P1_Feedback_Closure"
RESULTS = RUN_ROOT / "results"
WORKERS = 8
FAMILY = "all"             # "all", "timing", or "application"
MODE = "full"              # "check", "smoke", or "full"
PYTHON_OVERRIDE = None     # Reuses the original selected native interpreter by default
QOALA_REPO = None          # Optional explicit path to the existing pinned repo

assert FAMILY in ("all", "timing", "application")
assert MODE in ("check", "smoke", "full") and 1 <= WORKERS <= 8
assert (PACKAGE / "source/closure_runner.py").is_file(), "Original run package missing: check RUN_ROOT; do not delete the old run."
assert RESULTS.is_dir(), "Original results directory is missing."
LOGDIR = RUN_ROOT / "memory-recovery-v1" / FAMILY
LOGDIR.mkdir(parents=True, exist_ok=True)
print({"run": str(RUN_ROOT), "family": FAMILY, "mode": MODE, "workers": WORKERS})


In [ ]:
import base64, hashlib, io, json, os, signal, subprocess, zipfile
EMBEDDED_SHA256 = 'f40165f49cd0830ccafbe300433b9ffb74867b639098f0898c216b8439c98a5f'
EMBEDDED_BASE64 = 'UEsDBBQAAAAIAOkVWV2aTlBcdgIAAFEIAAAcAAAAYWRhcHRlci9MT0NBTF9WQUxJREFUSU9OLnR4dK2TwWrcMBCG734KsacWGiNblmXnWuilkEDoXSje2bWwLLmSnN1tyLt3tAmhWQJrQ32RGWnm+/8Z6XkDT2Dj5pZsdh5CLw/OD+AlHHXcfCObSW9xr0l/oAbpQ5CDfsRQWVa0wvBBGSMDdM5uA4aLvKx407ZtXTNaC17UcEP5S/a8iFPUn4E4Kz8DFWUlKKU1F7QsV1HYYkqbt1WFbhpRlaKpmEBOvZjTruE0HM2wmgouioKt4ZTlYk6Tty1DP5Uoy5ZXqyh8xWwo46lthahZVbbifTw7NWpzSqSoR233qfzotpAiYXQDpEDnxslAhESlZ6jdprO3hC0Vy4rFYmlOacMFaxqGAyibgnKxGEPXYWreFqxADOUNPzfkX69Jc3RRmVQZ/40K8aJCcZlTLsxZ5Gb560t1RSUKVic3haCNYJfS2EJpq+8E+3An6EsWASvDEbo5Oi/RIC6wlWoXzx47mKJ2lnyRclTaSpk/oIon8KfvPXRDyJflfyV5nhM3vOJ23v0BK4ObfQdS2wh7r+NJKruVk4ed0fseTyltZGdcgO0V/Op6H+WMMDp/Au/RwOTdpPYK4/KgY+/mKHtl99guibth7joI4Yqc1fU+ynEW5OtNw+S0LSdsZdfPdjg7OigdZZqpivpRGzR6Rc/6gheCvEbByqTpziPgMgf0gxi90zjrLrEmh32/1pkVld4lZDf/5cselCWcJBmBaItPqC54yLL7n1l2h8afgPyelY3zSPybctKrQO7ufxE/W9KDh5z8SOcgzQ9Qs92/1UMXxHl8VCpluRDJW6chEOwwgeNkdKejORGjHsHgjY4wTs6n8zt9jDM2JM/+AlBLAwQUAAAACADIFVldIFYY2rQKAAC4FgAAEgAAAGFkYXB0ZXIvUkVBRE1FLnR4dH1YTXPbOBK941fg5pkqfcROJpVKKgfFViba8YdGsjObkwsiIQlriuAApGTlML993wNAis7M7sEfEkGgu9/r192Yn8ub6c3d4ttwuljcLeRienn3dbr4JodDOf33/XRxO7nGP9PLh/vZ3a2cXE3m+FKI+62WTlfW1TqX2jnrpM2yxjl8VOtaO/n23S+nFZndVYXmf7XZmXIjs21TPnlpm1ratbgYvHrzbiTvt6qWxqeHA1lafCxzszd5owrpza4pVG1s6UdykueG/+L7tNdOHeVW7bVYm9L4Lc5a6bV1WtawtXI2097LylpstMWxuT2UPFJL68zGcCPXlCUsP5iikHvtzPoo+K7Kah7v1GFMP/ARR+rsqbKmrHEkzvJSlXlyV5oaH5t6i31rmLvXcL8p61GMWu1Uplcqe5La12pVmPS6VEVhs+CeXCtTNLDc4Ft5sO5Ju4FcWyw40K0jlltY5sQ/vHPYmgK/cTbjTFvaJ9G8kVxolfOZkn6HDeS/lkB2jbfEQfkQrfaN4OEHbEJUiAbCaNdYApzCL74lM9V4mLXTO+uOUj9v8ZkmjcS1LTfDAhHIkxfAlhwp64ILddbUGriVxxZb0CiiOZI3cTuna2VKnQuVOetbbtAeJasCJ5lVa0IkjJI53iw94ky+9SIE42kvcOhYfjO9/3J3RWC4Ya4qMhe7lhlB0SrbntgRTo6QrJ32W+krdYBlcn4E1mVLsZGY1HJnfS3/uFv8Nl0sZRlJ0EYALAMhyi4Qg8CdcFYG7HL69h9LnxOBBUFElNQGxiEejSsjTLCiKQDoJAFZbx2gjRzPrHVAGTHgWuu7FNCwkDzcaXwdcfGgfL1t/Xg9On8VMM+tjqg7DbjwhGeeFp0PbVkcxTxuO8eh04AoMNyp58da+Sf/WGn3GL3KAiZNxufKbZqd7lKCqHjZlDn8m58/ftY6Z4I8XhbWk4YKP6VGRkqNpNf5D2m7A0F++nkgYoDxn/S2cZmOWQodgee1zRiU9MUpffHB4zguUobiZcqqwbfCKyoWePld8wXmvTM+5NnOFKbNeHVAqucEyGnaQX6AitFCneIhfLNCRtRNoCHg1c9VYTKEWD+DcPTBOjKOnMWSATbLAF/KdViIw4Fk8ktQcsZwaq9LVWbkQaZNBSLc2pOvOwNS1CAVzvNNVTlinwerS3sSZNGTMkqvrUyIQm1xcLZV5YYC/wwcTULsU1RVkm+XZEQXOmOyrRuwMAQIrI4MzWAl0hg00itrn96L85H8Sm01OpJYPxsfpArC+gSKRwtjylSmZBrocm+cLeP5FxQwHK39mHEYRkzDVh0j/M4+aQomGDUSr/lGpVXto2jFpyGRu8h2hu5VYfKkqOH9mBsx41MKj8QbbvlnA8YQTFYEtXFa08K4XofNfjCorR4R3ZGcMuchB0JK6JQHeIr5GXyhBMXgp1h0VLOOGd/B62tb+c7+kfili8/LmERsgkvkF4QTMWcGhwo23EdM8n5uAOxbfUjG9wIT7fcQiCAPcBphCnUDZwx93eRHhHetoW8ZUuELAIYZmSrE7+cX49/PX+PnDX7exsymwjSoxbfR0WBolDYwt2YxLnXgZA4U8Qsx9qjz+RAhEShqBQp/KvVNRRMDmcBYmRXK7FBLpgGiNvwnqFJVIw8gT7l4cUTXbaBDwTLjj7LaHr3JGJfldIECIh+WUyEeqsJCdqFcMfCPEzZAj7GATdkbjUx1LFcSOfXXeH4eGA6ReEn/BPP8XIBipS5GclYHx3/AEZKBDoLZxkKiCko+LEPAaLWq34NNPGW8TjI6jDsPL15dvD1/dfFmuD8fi9+0rmI2LB5uHxd3d/cjeRXhzDWFoUXghVjCitAb4HDkTW1IK2p3rChtjku9W7Vmd3mViusHio869W5NiN3YoDawghEQmA2JF+JKrxUteN8W0o/vBvLm7mr68YwUORvIz5Ob2fW3j2d482wkP9kuT1O6RWVopU4kjmjD1qmvVTw2yBlLy9zpvbGNx9ddToS1qfNQQfvY8USv+90jmlv9Qse6hxGDx9ygaUDWjqpjWzC6RB+Jh8AJ4/8um5IBYm0HFR37Rhc4E4iB2ho7FplpmNnbT8wni8n19fRaLqfLJfr3pRCztfxDbdVKbpwioqp8kVjabLb1MGOcep0TiDudXH5Bmngf6lOrmyJ9E7tvEvJFCkEJKPP9bifJI4/pOiL2PgymYo5iXVan0qAcKi3aBQ9sqjYVka3wnbV5q1ix/9FkdGFZ6LoheS42V8nUM992qj0H0fceGDz8ZS2/nD+82Et8tkHgiM/BgmT2uy5b9wYhl1suxvnmjOXFlro9lGQQHV0rVv+wdViXmvlBDA0dR/PRI9YpQ8E348K5R4HRqeIYYxgvrKUhHW8Ku/Fj2M9m528VLbYuqOkcVhQaDzDTC/I6D1qU1Z0WhKpEWPupNviBdzSdTV10gy3EkUNd50hKMrVRZHHqRltTgszWqtQh5X7o7KDhsZwIxGBEf8glmNiwusVmB97LvIkRxYE1Uy82XhE6btgnJWT6NDs+LhNFu0RDMeQIYdahfIXKJDH01CFmmSoFwW5d7e1bdHwCJRPqQcJjBdsmZmHoBDjtVCTaDvFDYDXCpDB+mLUOQhLzOGTWCmeihbOVM/QyKRlW43nbHzRl0OG+fLQTja90BiVDBYaZbAaBfJr0IpWz2Gk/pgk4xByxi1cD8np2M7uf8AJgKSe3V/JqNvn19m55P7uEoHwOitsNF3KFrMtPTrwY2z7wzGPbN+zUk25zsSkLuM/engoV5aFPjSTsSYP1c8aOIDDsf2f1APU8Q3lMlfb/5T/o63U3sX18k8Z5loQuIUVoUkKAIzI9YcuOcfi87cYJ9hOBZKFotlPsy+5qRjFbzm5/vZ72/UvugYd71FrF+TZaPEiKSYCPWRGa74IqYzxc9c0aEBvNiT2eF2QguIKnFRMlVHy0wk8thSCg2ptN2d3JsJ90yCGgH9rTfuMLlEMXQCLPZ1dh52sk43PccrFcyp9+M59+Rn/aWZ5tnG0qgWkTZKyVTPTkcJMEDxWom4cOypWxsMHOLbSf8nQchgSEjjHtkQ6CoV5FZNLlTKuzgUVhEIohGxa0LFwPhPGGDSET1BacacH+nYi6vgNPkAFsC31UIVQ4CsA6KBGm7+L4nSnvTxUWU/I6BKNrMegiiaQ26DE3kVtoIjm5JkxbJOt019WbQQzKHBHhXIEgCAQmivyQX8H9WFPB6Dh30SR/urdorz/I+FizcwiGpElJBqmdzhYF7zjsZkCxrJu2GrAzppGxzrIaBLTTyAiyPXVXe0qkC6KRvGx76lRGYiPq9robCodJ/nvMH7dDajhbpGn9NNB+6M0849PtVRw6UtuUriPCWWlGizdx8PX67nJyLb9OrmdXQbUgUtxNl6gqWrtAMDRRxI0vr/sKJg3V3NSAaN3UONiPD8qQbluFvtANRK+v50uV2nRDe8idtk/5oWuPQh9HL83noS6LfndNL3gj6jR1mGYy7wa9WTCFqnfnyPAMKd+oJu3U7PS6oIK2DcOOhZOKTNXgdctzcCyAfHt3L/9s0Ao2u96Yn1iDNEp7tku60sK6yKK20rpsr/KoM8Ht3IEorf3oeDEFBgJGtM78yzb9dLdi2qEhjNbd6EsablJgQ7xG4r9QSwMEFAAAAAgA6BVZXYP4daXBAAAAFgEAACMAAABhZGFwdGVyL1JFQ09WRVJZX1NPVVJDRV9IQVNIRVMuanNvbjXMzWrDQAwE4Lufwvhcyv5Iq1VfxkgrbQmUJNgmYErfvRuaHgZm5vB9T/O83Ldb831fN29n+7pcP9/v5/IxLwSpJWrFgHIED5Y4GVKwsXj8EKqjoWBVVs5Sc4WS2MXFauawvD31od4evq1XOS4Pf9GJYs8U2bij6OC7kUgs6uZRu+doSaVEdm5eUTtgJVfyMCIV/ujD92N9+ee/LKYlixqCxwAFIFM1M3fO2SOS47NlBLMOZfzWWwP0jkqJeZl+pl9QSwMEFAAAAAgApBVZXfoaHyURBAAALQkAABwAAABhZGFwdGVyL3Byb2Nlc3NfcmVjeWNsaW5nLnB5fVXbbuM2EH3XV0y9D5ZSRe2ibwYEdBs4RdGgGzhOd98EWhpZXFOkyksc9+s7lChZNrw1bJgSObdzzgwXi8XzyTZKwi/Zx5/vS9V2zPKdQKg1mua+06pEYwDfsXRWaajp52TZMLnHig6pf1HCUekDapNF0ZqVDbRMsj1qsI1GVoE6Su+AlVacQEkE07GjJGta3w+m8DzEeVZKrEOoLPrDwjfFydg2zJ5T2CElgaDROi253AO3hp6MEzaDByUN/uNQ+mBk1Spj4Uskqao3HBMlX9zYFJisQCq42AQqrRJoyJKCUGTZ51w2Th6yaNtwb21RSyZA6bJBYzXZE4KsYh1tQKXIWirKuOKWYpzIoEPNW0oKjHK6xCxaLBYRbzulLZRKlk5r2s1qRzWRNTNQ1hGh2/pdSwEF38H5vH8TUB69fDNKjuuWoOCBOgJofK3MuKIYfR7js6Xkomiz/vRUPG8+P6xfXor11/XD6/bzBnJKJbvBTxRVWAOXb+qAcU2a8CCkTO9Nejj6v2QVAX2MZdpilfsYGeFQF6VyVIGOk37f6tNq4BJGL/Fd7+buLjjqz9WcIBenwan/fIAnLt07FeNrIAm4omXvmtTKJfzJf8ugZ4u+DCjupOWG75v7I/NUtUwf0plDzxqDFpkhGnq+VE0aIPqpMQRaJEHufjIeViKctMig3GvlOhC85TabXHWaSxt7TrLKtZ2JK17aGN/IZb7sW6sY5FaQFO0y7XiVK5Pt0dIqTs459c6QHQqqqzjwXT5S589qZ0gC8fRq8/ry6fd18bJ+ekyyCY5LZ0cCsTBIIqrMLU7uA19JktbCmSbfaofJQDY1QhFQjIMK0/9h/shtAzdFFVNmAQCTf0zbrgju8vCf+BYYO/5M+dDl+biRGbcj2ONBhLdTyQabILYP46QB0zhb0WiaqDXjYPF8K+flMbQjvacztG9c6U1rJwaig2qHAFFUCkZ+Hz25Icj6ogCPYFFwyW1RxAZFnc5R+IvAnQPRP08dcIaA1zC8W2nGDcL21OFaa6Xj5as0rvO9QLN1mpaq85CYFSx/pEkVm347Dm6TZPLrE8rG4Oc8vMivxolX3rgdL/thvkzm+c1d9Yd7SRUt0kVD4v4hD0ahgr+ZcGMJ/YZvWU1DnGusllcZDneKyWkmbfvlfCRdyGq2Tib8g2B69Ce5XE2budx6gueBR8nNWiGd13tbhDP+0bdZEECymkWYn3k/a+TAZZW+eYhSu5vldplUEHN8ZNz2HZuWTJYoinCj5N6Nx9VPOC+t5LrIRyYM3QG/Xt0uPqH5vDJxyEFpvvcTOf/O7RC0MB4bY98cB0EIGxpBNJAmNftrs7wQMxO+3hMNeJKUEJM8bueQ3+rF6c45cRTVxc3yHS9jCdF/UEsDBBQAAAAIAOgVWV0bIE3xlg4AAGwmAAAZAAAAYWRhcHRlci9yZWNvdmVyX25hdGl2ZS5web1a3XPjthF/11+BXB5Anikql2k7Gbt8UHx07IlPdmXfNanr4UAkJDGmSIYgbctp/vfuLgB+SPLlpu1UDzaJj8XuYj9+C/DNmzdzqZqNZPVaMvmcqjrNV+z6HVumzzIZz5iqmySVij2l9ZotiiZPZMKeiupBVixLl7JON9IfjWYFW1bFi8xhYiZZqlgln6q0rmXus6sqXaW5yFia13IFrVuPlVVRF3GReSxey/ihLKCPiTwZ5aJOHyWrxJPuUcBCmmVMlGW29dlZJdWaqU3xIFlcbEoR17CUarJasU2jahACmrIt24g6Xo9QLCUegefCMqHn2jkLuSwqyQRbNrAISrvV3KNWEn/05s2bUbopiwq4q1alqJS073EB4jzXWbqwLcs4rzP7shZq3evT/3oNv6git8+FGoH6NqwUNc4xg9k1vNohKl0B9+1bswANxlKptmXbPuKetM+ViOVCxA+24SUtcYvMeppGVMl4G2e49VYSVHOk91mNRufhPAyQGyeKcHYUuT6MKLJH6bg+aEXm9WiUyCVTa+GU7vGIwW8dGB340Prtn//iuNRMpkTEStcvSpk7vFpwlwnFlnoi/mBbGCgiZ2ktKycTm0Uijpewqkicd998+6e3+Mf1Fpy7x2u/KRNRS2ehV6hk3VQ5W/tr+ZykK6lqWJrYQ5uUTuk9GxbLwDByUhox/M1DklaOflHBbdVIjxwjKh7ozT2pN2VQ+ihGlIsNkPPx3xH3oYNrDuDJp7UiNBEH99pPmk2pnGcvBR/K6+BbT4Gmowe51au4R/yfOSfqIGWZwb4BX5ptkhrUagQjclkhEuVYPeIIvZZr5mzkpqi2kcwf06rIN7CkY4QGVYngt99HVs3IPWra4XqOvxHP3LMvcVOhKrqGUoqH7k0+op642+2c0SmfgElOlmoSr6qiAcVMcJ12VLpkpZ8qMienN7u1kM69fNWUJZibcq5uwqoqKvcYRbhTNeyTex+Ufel9aE1LY2pfs1sIAMbKISRswd+zAsy8LsDlczAMiAyaP5/NZVxUCUXCDUQb9IYKBIFAV69FbujFmUg32AVtNY1dp7ISVbzesqooaq4gLG7SGoMIBR9YOC1yaIaw1ED4oV5/NFAUMjhRMlu2uhodUBHJzK3LmpGfk15PUDH4GIzj00eRZmIB4VnPBQupBY456WQYowzsUWSN1PrK4XUh+6IYCXjf05CKtjobZqNNkTSZVOBI8YNYSbPDYBI+Bjk/zZWsaucbjzZRj5lwVTRVLLlr3Dhog6avnwxVh8dZoZpKRlWT57ICr0n+eGwiMYZ2uiXlV/6BiPZVsMPRZLieX275cSVSJdknVBUZpcM/5vK5lDHZ1GC8iap8EJ0qL9Eqe5RVutxGeqVdhaF/QmDW/tlydXM+hXh68/HDjV8/A92hCZRZWuMc1fcry5qHXhhgtx7ocMa4987tOyYFcbMWDgeF2OkHxG7Tu5mSaAyQpMslJI9jxo+IxqjPRUARDdPKhM/D06tP4fzn6Obq4/w0jM6nN+fhjY9BzmhMu07wG9kKyJoRRojqgii47rHOOqSsEjWFkVVq+v4qKxYOf4t75rq/283XJD8rF8YD2JstAycpIQkxvUNWsOFuanp6Q2ORZZF1BKfyADg82nxTyce0aFSAjoDtOltU226jbE9A8W3XSF3vLXbet8MpVg7StdPmiQoieZqbaLBEbrLtcUvf8mISN+xRRNgIOE68pdik2dazUni0ghHia/Z9AYsKaN8qiHMorzJmTJGixVoW5SmEdojpJhazITKQSsfByidSL9KxE92Trk2vTANjwJ918E3rGL8UhBASHx6UY1jmJEQ/IRUZ2JuPK2cSzaIVCqa5J7l8GvTSetTVdwmggSF9VuQoHqSOJ/t6wHI+pBApIUcYmNoCXN6RFLAm+YBlZgIr3HGjHn7vnizsCGJoAiv2u/u8ia+CxQEuCCqPbfIbQuaecyaQMzArOChyR5d0fRRkAM/EwNKTNK4dmFKDFfPr6c1NFP40Pb2NTq8+XOP/D9Pb03NuDCgwm6JRfSSfZdzUkERUQPS9bo+MGgI0+tYMtNlRmzaD1iAja1qRxpYBhoBWl9z2mijSraM95dBkrebXZ1IeDSAqAP7BEG8KFdKrR+5jSyATJ9Dk07xsanXCmjwGDIGxEQsbATVVTRYsRbxmkCR8MCWsfxiCQ4DEiBAB8m8aDHWQc4slZOJUbVm5BpeLlc8NyssEkoYtMvFFh+BA+OapS2onpuSBPvPU69PCCV9vF5HKihUA4cCO1YB4YmDfuDLBcfz4DoAezu5N+mMMrcGAzHQycHhNoApQJZZ5aUxCQ0mAuYg4CzjEGM5kBjbuUJNnaGhDHBjlze10fhu+NzYI1WtgV/IAE6CY+M8z4RJezVO317ohsgVuwLXbm6gWr5v8gZWQEZa6Gi3FUw5ba1yN960NDcKmdkQk6TIFgc8ECOKBzUDMiVT6st/XkQDQhkjCmN8HUj95OIOgJyuobU900cvarYGqm6wmFo2iYhxRnCQ3TyHsJiYO6WJIb9kEtXb70aRdT6uyV6+1w2y2hvojfuC2gBO6fsO2XgmH5bC/xEYH/3i64fLq9EeIGf/qvc2+7wLPIBnSFmvtBa/gpMFYSF3BqxB0MFIX/hBjNb1IJyRnOGioofCn8PTj7cXVLLqegw5m09lpaPRF9mdwguGwjS2aX29AeM847GjNlldu63WRE0rQMRMjtAcaQAROzeZ5n+x+3RccKgU/Z/4dKeAiCXj4DGKh5WtewPRUs1B1Wjc6OOXZ9qRzD405PONE+NTCAIiTUiYaDVAQhMcKse0jBNGki5I+d4cbQTkUNgswyzJLV+u63RQdhsdth9kQDHJlMSQC0YToQE1E5s3vv9IZzIL5OaQkcHeTP2c6wLeU2QKNGKAiJE3GTxgH2AGSGpISJwHJHcYxyENm16diObsz4XTCe5sx2WX+bTvMWR7xMR5NQfnMd4YR7SXBXRPe7u+P9/YwXZL3Gx58CsMIEbG8TvNG7k0gtETIRE9xD5GEPn8loXgwmnRbTb5Kd3jSkDcbAOQez2Wtfm3SBB7LbbLNhYCnXwuRCd35q1Ab7u6L1fEBqIjOCfk9cWTqFbMr+12HSeFvH0EZC9BxBK34UeYijxHS6WQOKC+WQ8gLyXxY8vS41YkHkhmxxY+18jpzNHhqNr29+BRCkAnPLi9+OL+NrmaXP/MTjb+G4Q5sPLjj4zE+cKqljeHf6/XwWWfNu/vBTCjx00RoKLbTRWZFOKBvW/t6ayFbZ6w0CyzWQPC9KV+z0J4x0xDt/uawdYIheEyqgUwNEBtCHEOQYQ6SD1Cj1CRMGgbNMzyfrrrtodDq703cLdDuOMwFexuPtQQWvWIL7hi3RQU2mGhptW1eXZoN5igr3dNh2Le4Cff7ytDYdieOdfvCv0Cf/w9BDPB+TYqeIfkA3gC6Ol9USu5T2oEjH65+DKPwbx8vPk0vwy7T9tbbJ1FWUGX1T1t7o+/G7+6hoMgaUHqHQe2v751aRa9559k8vDlHBHAawptm81UH/VJf0go/4Ek69h8wfTzY7KqKfvhxXF2L47WNpAOMXuF9gNLwAkbrUDHKzQ3kf9gVcyWjb4qocPuf+BTJ9jlLPBiqd82TnslqjWlj66sh6bAh6y22twiD4vbs4qfwfTSzQflidhvOZ9PLaPrx/cUt93r1WgRqjujkfgtzw9n7i9kPva2Tz7Esa/a9UDKkR4ROgJmh/fgLmDmbXlwCI5AUbsI5MHJ6Hp7+eH0F/Nxwj8BHVG+hQsA/DtB0/YhuJ6LI20BRgkUhKgZ7vPZSKGiffLDUjaihQo+dHd+kvNiBenOMtOs9vOJo4gCRoH4BE0RT4hrNDjr2bd/62RAPazwetSYO/reH1QMDlw+o784UqhHJlNYw/RAKPvmCGojqbPlMZ8p4ZCafrHl5VJQZkWx5rU/s9EvvTPnkP6qn2wr2vyiR8UgfkrwzwJT8CJ0Q4mH33OFJuyq1YJ/GbDZ16Jh03xG3gd/o0eOtNPxIq8ic41bxGiLJrg4sZ+YazU6Cp13laPX7L2l7RZLj9ROMsSCGEIG54/T/kZZneHdi1vX4E/favovr6D0ArOlt+J5q15fh3SMK5iHiTp9RepJzz+gRWWPPH8Dqh7x4yu2NW3+RXug1a/iVPq/uetTkrTmL2ncdUIxMy1rj9W7KQcjOITyRJ5pJx8QVHTL3ptpbULM1ZuwdTb7vmbML7tjrM+Ur30eS5a5kbw8JYnTZu+7CU1Z9seqDaSncV8fcsB7u0ucRr/Wt+GdKnkpmwfBeARl2T+iqRNvAEZ+AWVJmyf5IvRlqD9InsMLN8Y3a79JcgttOLy+jq7OzcA7h3Ryi0ob7qxf+SuFjrd46Hh0/4L1agDS9xbaGkFHiwa6+E6QzJs8cMdC+U0Fk71PsnpL29F2KB2AG6niZRAWEZDxmqjRq2hceI87xi2+v1oeVjwYqqPmqyDL8aKRY4YlU0dQqTah2gnhF3gVw3UYIE24BVORgFj1iIo5TzLWYgkANcdYk+PGGggAB+cYgDQAvOZ57r/DOGBecGLIdaoHGFnmfXs1u51eXl+Gc7KR/sg7vnWtbEaFxP8KBfezQ6XRgJqK2+Ty8nV7MYKfn079HH6azi7Pw5tYknB5ubXe4/VrA3oma6yYtkLnppwMX+0GD/UzFn1arBrPcNb5VDn7iIJIkEqbZwduEDaRp7sXrIo0pyWjMprVvz2B2p43H5kCNe4Q4MOGBuUAVX8nEfCKxP8fE+NfnvLJWiw1pHkQoD0QWQCr47tAyFmx2IgkCmYezZ0uLRr3GgoarHUVdvXs7+bCjpd8P6gCr9FYBJikG9OmJkjjUHkCaePjur93x3F+D7w7dOcP7O/CiqmhWa/adOXRTdDot8axoHOP3TfYzhJa68M3uQwAy232sq6ch0mkvCzx7OeC6/UKnvX8YjYCqhZxANIrQLKOIH5s7yH8DUEsDBBQAAAAIAM4VWV3JW9sn9gYAAEkSAAAYAAAAYWRhcHRlci90ZXN0X3JlY292ZXJ5LnB5tVhtb9s4Ev7uX8FLP5DC6pSmix4OKfQhl3WKYrtJ4KaHxQUBQUsjmw1Fakkqtrvof7+hRPk93ewtzh9iUyLn5ZmZZ4Y5OTkZ65nUAFbqGank0rcWHDFarTJyNwcHxIPzjpSGaOMJLKFoPZDfWqF9WxMn61YJL4122cnJyUjWjbGeCDtrhHUwrAuji9Za0D6r2l6FcKSohvf9l5LT4QFK9bKxpgDn0LLhsXGjypqaNMLPcXc8R25xOWxxKzf89FA3lVRrK1otffCmlzGsstoUj4MkFFysRUX93EKxKlQACI1eL4Zd+MA8geUaYXiCuCU8WY1GtxeXP1+8H+fBQGZcBvpJWqOzGXhGb8/41eTmP+NrHrfR1HnLur2cB8M5TzLEyqgnYEmGiCKA7v7s4ZT2yrjzwnq+MPbxNEgDKKeieOSXyjgEmSbJ1vnRqISKyBJlyGrFlsm5BQyFJqUsPHsSqoV8mTayzNFQNBB/sSTpDlkhHfAaamP7g2FNfunWY2uNZbQEjAdYgbkBhxlFo/a47CwGy76YaVoZVYJNzkcEP6/IeNkoWUivVhhJ7efgZUGUrDrUYRBApCZ3EPAXdvWTRMA9mtKnbSfIpmU+hCEzVqJFQvHalK0CxyLeybteeR+eaMi7R1jlZYbIThUEC5NeoFjkFP+c0h9www80++JCHL/Sd4WpG1H4nMYfBzv689nCSg9RyylKSjvcH6Uuc3pzdfXxw/WYj6/f49d48uH6Pb/68Ovd58kYg/hu73BUlN4/RNugANn4vBOIFucB12Bw0DKYNxzCZ9zNxZu3/8htVsoZVsCWUcmw/7k98XWSxgzsCSHUf36WLoRS3CHsunR59vr1GSa08K37AwePAUSLORSPjZGY8fSUbRClSRr9RVz6BI7r0ahQwjkyiXG/DBIcW9f5Hf65FA5iroV8DM+50cArLJM5Hwq+wXIu5q1+5EKXfCGk553fXk6lkn7FHKgqigmfhfTzDTFkV0HYbS9r3OGDFVKLZUx7l79J6wYlag9Ln+9RXSi94R2jrhELdDkJtNIYozY6wydyaX4fXmWundbSs6HCU5mQylgiQ7FYoWfA3iYPO+dLdD1tQJeoNy+qLHjKotBkZ2dwOENwwforoRyweApjYBZoQBV4Bh1hvc4q6IyCNiq3hIyxgSh2b+9pRzz0oTtmO1NR4EOqJObcYHWSPC9DgWa/oxykq30p35L07dGTd7YFhrnKhnN/2+a8XSGoezddehqEQHshYRoxQ9JzPOSAaT2fi0B/M45vXVuEoP7VbDn7bvTz7dhvE3X6ejeEndotECZhr5vADJZsi8pTepS/z3ttQ5i9rAG9zX98vQ8PRBewazr8gpKLymM9wbKAJhDFPhxTQLwhpF8E4RYdGoAYvcT6f4cUisZvDD50foN5X+4RYLa3dy9ZPjh23Lb0+zFMDoT2bXNjLdtYeyxLn1fcQ7aP/LrNhRjVgF8tQsSRCmUlMQ5bjLofg5e1y/V27C9Y8127GTo6tjFY5oeU8+MW5cRI28yZ1haApeJwwmTJbpCHqS07bPCsqwRfN2k/qGVm+gXfsDKlwSSa9g2B98MMdpNuxMAS3tuPedLHnqa7E8mRjY2FSsnZ3O9J/532vY2e09uLT5/otyNnBaK4+grPn/xeW/z251LYZrbVjGJh4luaUlebR9TcjTaIWJK+2U3I2DQxjjYLzU2Bh5JttofJpwsm/gjhDEA+PNcTNoQapSa7yl5hU3aAE/kccEiDv7s53iXWs5wPcyOmdXitSS+R9ElMlGg1Jq7bE/cVrBnm844aHVnMMWnwmBdSB+oKumCJbSQsYIlDy9rn7BDYv5ZPQ1RSOhlffOS3k5vL8adPfPzr+PLz3c0EB3vsyhwqHGN9ftE5iEwYSWDSeyp8yHwMAvEGLcVoEkGGyJBuHMFB4JCqXpHP4ZaG3opS4HmLA32oF7U6J3GOID0Xd/c00tEAEVMs9iOykB3C+GAwc3EEt/BbK8NtLYi3IBQZ6J3iJTHyRLxT+FV2IO8Q2qJK6RFKo39Epn86v48MGy/P83Sdx88PHgc09gwt42XzK2geNwd8ZzjorrrRck0uvBJS8QLvbWjcPjkPxNwx+SpKOkLLL7z0/N9o+GVs+a+PN5c/j3/CEHYzFD66H/ogqWU3AyN4Yb4EXazowyGxRhdTWoT7xuAo7azq+WK3SoSduXz4l0R2LWrAsbrAGRbvymIGeUQm7Wcbl6+T45TGR9gnRC3VKl/nH6IKOa1apWg6DGr/RAmNya+R4V4+ek1aHYapOL6s8SNTZYpHKGm4pseQ9r6x4M6u/P5FFM61wQsEIoOZ9L+lr6wI5xph4jzPKec1Mirn9HzzHxN8wNCkqXGYyTlW3n8BUEsBAhQDFAAAAAgA6RVZXZpOUFx2AgAAUQgAABwAAAAAAAAAAAAAAKSBAAAAAGFkYXB0ZXIvTE9DQUxfVkFMSURBVElPTi50eHRQSwECFAMUAAAACADIFVldIFYY2rQKAAC4FgAAEgAAAAAAAAAAAAAApIGwAgAAYWRhcHRlci9SRUFETUUudHh0UEsBAhQDFAAAAAgA6BVZXYP4daXBAAAAFgEAACMAAAAAAAAAAAAAAKSBlA0AAGFkYXB0ZXIvUkVDT1ZFUllfU09VUkNFX0hBU0hFUy5qc29uUEsBAhQDFAAAAAgApBVZXfoaHyURBAAALQkAABwAAAAAAAAAAAAAAKSBlg4AAGFkYXB0ZXIvcHJvY2Vzc19yZWN5Y2xpbmcucHlQSwECFAMUAAAACADoFVldGyBN8ZYOAABsJgAAGQAAAAAAAAAAAAAApIHhEgAAYWRhcHRlci9yZWNvdmVyX25hdGl2ZS5weVBLAQIUAxQAAAAIAM4VWV3JW9sn9gYAAEkSAAAYAAAAAAAAAAAAAACkga4hAABhZGFwdGVyL3Rlc3RfcmVjb3ZlcnkucHlQSwUGAAAAAAYABgCyAQAA2igAAAAA'
data = base64.b64decode(EMBEDDED_BASE64)
assert hashlib.sha256(data).hexdigest() == EMBEDDED_SHA256
RECOVERY_DIR = RUN_ROOT / "memory-recovery-v1"
with zipfile.ZipFile(io.BytesIO(data)) as z:
    for item in z.infolist():
        rel = Path(item.filename)
        if rel.is_absolute() or ".." in rel.parts: raise RuntimeError("Unsafe recovery archive path")
        dest = RECOVERY_DIR / rel
        content = z.read(item)
        if dest.exists() and dest.read_bytes() != content:
            raise RuntimeError("Existing recovery adapter differs: preserve it before using a newly versioned adapter.")
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists(): dest.write_bytes(content)
ADAPTER = RECOVERY_DIR / "adapter"
print("Verified recovery adapter:", ADAPTER)


In [ ]:
# Prefer the exact interpreter used in the original notebook.
previous = RUN_ROOT / "PYTHON_PROBES.json"
prior = []
if previous.exists():
    prior = [Path(x["python"]) for x in json.loads(previous.read_text()) if x.get("ready")]
candidates = ([Path(PYTHON_OVERRIDE)] if PYTHON_OVERRIDE else
              prior + [Path(sys.executable), PROJECT_DIR / ".venv-p1/bin/python", PROJECT_DIR / ".venv/bin/python"])
probe = """import importlib,json,sys
errors={}
for n in ('numpy','netsquid','pydynaa','qoala','netqasm'):
    try: importlib.import_module(n)
    except Exception as e: errors[n]=repr(e)
print(json.dumps({'python':sys.executable,'errors':errors}))
"""
PYTHON = None; attempts = []
for candidate in dict.fromkeys(candidates):
    if not candidate.is_file(): continue
    result = subprocess.run([str(candidate), "-c", probe], capture_output=True, text=True, timeout=60)
    try: item = json.loads(result.stdout.strip().splitlines()[-1])
    except Exception: item = {"python":str(candidate),"errors":{"probe":result.stderr or result.stdout}}
    attempts.append(item)
    if result.returncode == 0 and not item["errors"]:
        PYTHON = str(candidate); break
(LOGDIR / "PYTHON_PROBES.json").write_text(json.dumps(attempts,indent=2)+"\n")
if PYTHON is None: raise RuntimeError("Existing native P1 Python not found; select that kernel or set PYTHON_OVERRIDE. Nothing has resumed.")
print("Using:", PYTHON)


The next cell runs host-process engineering checks, then the native recovery workflow. The original runner prints the number of verified completed chunks and the remaining chunks. Fresh workers print PID and peak RSS in KiB. These measurements do not establish a specific memory leak.

If eight fresh chunks still exceed the session's memory allowance, increase that allowance or set `WORKERS=4` and resume the same run. Do not reduce the sample size or delete successful checkpoints. Separate sessions can use their own eight-core allocations concurrently.


In [ ]:
from IPython.display import FileLink, display

def run_stream(command, log, env=None):
    with Path(log).open("a", buffering=1) as out:
        out.write("\nCOMMAND " + json.dumps(list(map(str,command))) + "\n")
        process = subprocess.Popen(list(map(str,command)), stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True, env=env)
        try:
            for line in process.stdout:
                print(line,end="",flush=True);out.write(line)
            return process.wait()
        except BaseException:
            try: os.killpg(process.pid,signal.SIGTERM)
            except ProcessLookupError: pass
            try: process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                try: os.killpg(process.pid,signal.SIGKILL)
                except ProcessLookupError: pass
            raise

test_env = dict(os.environ, P1_FROZEN_PACKAGE=str(PACKAGE))
code = run_stream([PYTHON, ADAPTER / "test_recovery.py"], LOGDIR / "ENGINEERING_TESTS.log", env=test_env)
if code: raise RuntimeError("Recovery engineering checks failed; see the printed test output.")
command = [PYTHON, ADAPTER / "recover_native.py", "run", "--package", PACKAGE,
           "--results", RESULTS, "--workers", WORKERS, "--family", FAMILY, "--mode", MODE]
if QOALA_REPO: command += ["--repo", QOALA_REPO]
exit_code = None
try:
    exit_code = run_stream(command, LOGDIR / "CONTROLLER.log")
finally:
    export = [PYTHON, ADAPTER / "recover_native.py", "export", "--package", PACKAGE, "--results", RESULTS, "--family", FAMILY]
    result = subprocess.run(list(map(str,export)),capture_output=True,text=True)
    print(result.stdout, result.stderr)
    status = LOGDIR / "STATUS.json"
    if status.exists(): print(status.read_text())
    log = LOGDIR / "CONTROLLER.log"
    if log.exists(): display(FileLink(os.path.relpath(log,Path.cwd())))
    review = RUN_ROOT / (RESULTS.name+"-"+FAMILY+"-memory-recovery-review.zip")
    if review.exists(): display(FileLink(os.path.relpath(review,Path.cwd())))
if exit_code:
    raise RuntimeError("Recovery stopped. The exact status/exception and controller log are printed above; preserve the existing checkpoints.")


A successful full recovery means the unchanged native checks and fixed-N studies completed. Independent replay of saved raw data is still required before updating paper claims. Keep native raw files on the server and return the compact review ZIP.

If this notebook stops, send the final `STATUS.json` and the end of `CONTROLLER.log`. A new memory failure in a fresh worker, a parent-analysis allocation failure, a source/environment mismatch, and a smoke-result mismatch require different fixes; do not treat them as interchangeable.
